# Publicando uma tabela no catálogo

## Task 4: publicar uma tabela que o time consiga consultar pelo nome

## O problema

As views somem quando a sessão termina. A task: **publicar uma tabela** no **catálogo**, o lugar onde ficam os nomes
das tabelas e os seus **metadados** (colunas, tipos e onde está o dado).

A célula abaixo já vem pronta: o mesmo código do vídeo 3.1, com as cinco views do projeto.

In [ ]:
# vídeo 3.1: as tabelas do projeto
from wc_pipeline.schemas import MATCHES
from wc_pipeline.transformacoes import limpar_partidas, partidas_por_selecao, resumo_por_selecao

partidas = limpar_partidas(
    spark.read.option("header", "true").schema(MATCHES).csv(f"{RAW}/WorldCupMatches.csv")
)
por_selecao = partidas_por_selecao(partidas)
resumo = resumo_por_selecao(por_selecao)

# vídeo 2.4: escalações e Copas
from wc_pipeline.schemas import PLAYERS, WORLD_CUPS

escalacoes = (
    spark.read.option("header", "true").schema(PLAYERS).csv(f"{RAW}/WorldCupPlayers.csv")
    .select(
        F.col("MatchID").alias("match_id"),
        F.col("Team Initials").alias("sigla"),
        F.col("Player Name").alias("jogador"),
        F.col("Position").alias("posicao"),
    )
)

copas = (
    spark.read.option("header", "true").schema(WORLD_CUPS).csv(f"{RAW}/WorldCups.csv")
    .select(
        F.col("Year").alias("ano"),
        F.col("Country").alias("sede"),
        F.col("Winner").alias("campeao"),
    )
)

# vídeo 3.1: as views
partidas.createOrReplaceTempView("partidas")
por_selecao.createOrReplaceTempView("por_selecao")
resumo.createOrReplaceTempView("resumo")
escalacoes.createOrReplaceTempView("escalacoes")
copas.createOrReplaceTempView("copas")

A próxima célula também vem pronta: ela só limpa as pastas da demonstração em `/tmp`, para o notebook poder rodar de
novo.

In [ ]:
import shutil
for pasta in ["/tmp/spark-warehouse/copa.db", "/tmp/copa_resumo_externa"]:
    shutil.rmtree(pasta, ignore_errors=True)

## View ou tabela?

| Objeto | Como cria | Grava dado? | Vive enquanto... |
|---|---|---|---|
| temp view | `createOrReplaceTempView` | não | a `SparkSession` existir |
| tabela | `saveAsTable` ou `CREATE TABLE` | **sim** | alguém apagar (com uma ressalva) |

## Publicando uma tabela: `saveAsTable`

- **Type: MANAGED** (gerenciada): o Spark é dono dos arquivos.
- **Provider: parquet**: gravado em Parquet (aula 5).
- **Location**: a pasta dos arquivos, dentro do `spark.sql.warehouse.dir` da sessão.

## Gerenciada ou externa?

Tabela **externa**: criada com `LOCATION`, numa pasta escolhida por você. No `DROP TABLE`:

- **gerenciada**: apaga o registro **e os arquivos**;
- **externa**: apaga só o registro; os arquivos ficam.

Registro apagado, arquivos mantidos.

## O que sobrevive quando a sessão termina?

Simulando o "dia seguinte": encerrar a sessão e abrir outra.

As views, o banco e a tabela sumiram, mas **os arquivos continuam no disco**: no curso, o catálogo fica **em memória**
e morre com a aplicação. Dá para registrar a tabela de novo, apontando para a pasta:

O dado voltou inteiro: só o **nome** tinha se perdido. Agora a tabela é **externa** (tem `LOCATION`).

Em produção, um catálogo **persistente** (Hive Metastore, AWS Glue Data Catalog, Unity Catalog) guarda os metadados
fora da sessão. No projeto, a aula 5 grava o resultado em arquivos Parquet numa pasta.

## Demanda da aula 3 atendida

| Etapa | Entrega |
|---|---|
| Task 1 | as cinco views do projeto em SQL |
| Task 2 | estatísticas em SQL idênticas às da aula 2, e `sql/gols_por_copa.sql` |
| Task 3 | a seleção que mais marcou em cada Copa e a comparação com a anterior |
| Task 4 | `copa.resumo_selecoes` publicada, e o que sobrevive à sessão |

## Para consultar depois

- **Global temp view** (`createOrReplaceGlobalTempView`): uma view que vale para todas as sessões da mesma aplicação,
  no banco `global_temp` (`SELECT * FROM global_temp.nome`). Some quando a aplicação termina.
- **`spark.catalog`:** `listDatabases()`, `listTables("banco")`, `listColumns("tabela")` e `tableExists("banco.tabela")`
  mostram os metadados como objetos Python.
- **`spark.table("copa.resumo_selecoes")`** lê uma tabela do catálogo como DataFrame.

## Encerrando

Ao terminar o notebook, encerre a sessão Spark. Isso libera a memória e a porta da Spark UI para o próximo notebook.

In [ ]:
spark.stop()